# ISOM 839 · Session 2 — The Art of Formulation

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Last week you solved a model we gave you. Tonight you learn the most valuable skill in this course: **turning a paragraph of business prose into a model** — decision variables, objective, constraints — and only then typing code.

The rule of the night: **formulate on paper first, code second.** gurobipy is a transcription tool; the model in your head is the deliverable.

> Run cells with `Shift+Enter`. The free pip version of gurobipy solves everything in this notebook — no license needed tonight (but claim your academic license for Homework #1: [gurobi.com/academics](https://www.gurobi.com/academics/)).

In [ ]:
%pip install -q gurobipy
import gurobipy as gp
from gurobipy import GRB
print('gurobipy ready:', gp.gurobi.version())

---
## Part 1 — The Shop Floor (production mix)

> A workshop builds **desks** ($70 profit) and **chairs** ($50 profit). A desk takes 4 hours of carpentry and 2 hours of finishing; a chair takes 3 and 1. This week the shop has **240 carpentry hours** and **100 finishing hours**. What should it build?

**Before any code — the three questions, on paper:**

1. *What can I decide?* → D = desks to build, C = chairs to build
2. *What am I maximizing?* → profit: 70D + 50C
3. *What rules can't I break?* → 4D + 3C ≤ 240 (carpentry) · 2D + C ≤ 100 (finishing) · D, C ≥ 0

Now — and only now — we transcribe:

In [ ]:
m = gp.Model('production_mix')

D = m.addVar(name='desks')
C = m.addVar(name='chairs')

m.setObjective(70*D + 50*C, GRB.MAXIMIZE)

carpentry = m.addConstr(4*D + 3*C <= 240, 'carpentry_hours')
finishing = m.addConstr(2*D + 1*C <= 100, 'finishing_hours')

m.optimize()

print(f'\nBuild {D.X:.0f} desks and {C.X:.0f} chairs -> profit ${m.ObjVal:,.0f}')
print(f'Carpentry slack: {carpentry.Slack:.1f} hours unused')
print(f'Finishing slack: {finishing.Slack:.1f} hours unused')

**Read the output like a manager.** The optimum is (30, 40) with $4,100 — and *both* slacks are zero: every carpentry hour and every finishing hour is used. Both constraints are **binding**.

So here's the question you cannot answer yet: *if you could hire one more hour of labor, which department should get it?* Hold that thought — it is exactly what shadow prices answer next week.

### 1b — Stress test: a demand cap

Marketing says at most **20 desks** can sell this week. Copy the model, add one constraint (`D <= 20`), re-solve. Watch the plan — not just the profit — change.

In [ ]:
m2 = gp.Model('production_mix_capped')
D = m2.addVar(name='desks')
C = m2.addVar(name='chairs')
m2.setObjective(70*D + 50*C, GRB.MAXIMIZE)
m2.addConstr(4*D + 3*C <= 240, 'carpentry_hours')
m2.addConstr(2*D + 1*C <= 100, 'finishing_hours')
m2.addConstr(D <= 20, 'desk_demand_cap')
m2.optimize()
print(f'\nBuild {D.X:.2f} desks and {C.X:.2f} chairs -> profit ${m2.ObjVal:,.2f}')

Two things to notice:

1. The optimum jumped to a **new corner** — (20, 53.33) — and profit fell only $33. The model absorbed the shock by rebalancing toward chairs.
2. **53.33 chairs?!** The LP happily builds a third of a chair. For planning rates that's fine; when the decision must be whole units, we need *integer* optimization — Sessions 6–7.


---
## Part 2 — Harbor Fuels (blending + the ratio trick)

Blending is where LP earned its first industrial fortunes — refineries have run it daily since the 1950s.

> Harbor Fuels must deliver **10,000 gallons** of Regular gasoline (octane **at least 87**). It blends two stocks: **Stock A** (octane 93, $2.80/gal) and **Stock B** (octane 83, $2.30/gal). Minimize cost.

The octane spec is naturally a **ratio**:
$$\frac{93A + 83B}{A + B} \ge 87$$

Ratios of variables are NOT linear — solvers reject them. **The trick: cross-multiply** (safe because A + B > 0):
$$93A + 83B \ge 87(A + B) \quad\Longrightarrow\quad 6A - 4B \ge 0$$

One TODO below: write that linearized constraint.

In [ ]:
mf = gp.Model('harbor_fuels')
A = mf.addVar(name='stock_A')
B = mf.addVar(name='stock_B')

mf.setObjective(2.80*A + 2.30*B, GRB.MINIMIZE)
mf.addConstr(A + B == 10000, 'demand')

# TODO: add the linearized octane constraint  (hint: 6A - 4B >= 0)
# mf.addConstr( ... , 'octane_spec')

mf.optimize()
if mf.status == GRB.OPTIMAL:
    print(f'\nBlend {A.X:,.0f} gal of A with {B.X:,.0f} gal of B -> cost ${mf.ObjVal:,.0f}')

**Check yourself:** with the octane constraint in place you should get **4,000 gal of A + 6,000 gal of B at $25,000**. (Without it, the solver buys 100% cheap Stock B — always sanity-check that your specs actually bind!)

Note the elegance: the solver uses *exactly* enough premium stock to hit octane 87.0 — not 87.1. Quality specs are met, never gold-plated. That discipline, at refinery scale, is worth billions a year.

---
# Homework #1 — The Blending Challenge ☕

**Due before Session 3 (Wed Sep 23), on Canvas: this notebook completed + a five-sentence plain-English summary.**

Beacon Hill Roasters blends three bean origins into a **100-lb batch** of its House Blend:

| Origin | Cost / lb | Caffeine % | Flavor score |
|---|---|---|---|
| Brazil | $4.20 | 1.3% | 6.5 |
| Colombia | $6.40 | 1.0% | 8.2 |
| Ethiopia | $7.60 | 0.7% | 9.6 |

The House Blend must satisfy:
- exactly **100 lbs** total
- weighted-average **flavor score ≥ 8.0**
- weighted-average **caffeine ≤ 1.05%**
- **minimize total cost**

*(Because the batch is exactly 100 lbs, the weighted averages are already linear — no ratio trick needed. Convince yourself of that before you code.)*

**Your tasks:**
1. Formulate on paper: variables, objective, all constraints.
2. Build and solve the model below.
3. Answer in one sentence each:
   - Which spec is binding — flavor or caffeine? How do you know?
   - The solver uses **zero pounds of Colombia** — the middle bean on every dimension. Why does the cheap+premium "barbell" beat the compromise bean?
   - If the flavor requirement rose to 8.5, would cost go up or down? (Answer without re-solving, then verify by re-solving.)

In [ ]:
hw = gp.Model('house_blend')

# TODO 1: decision variables — pounds of each origin
# brazil = hw.addVar(name='brazil')
# ...

# TODO 2: objective — minimize total cost

# TODO 3: constraints — batch size, flavor spec, caffeine spec

hw.optimize()

# TODO 4: print the blend recipe and total cost, and each constraint's slack

---
### Submission checklist
- [ ] All TODO cells completed and running
- [ ] Three written answers (binding spec · why Colombia loses · flavor 8.5 prediction + verification)
- [ ] Five-sentence plain-English summary of the optimal blend, as if to the roastery owner
- [ ] Gurobi academic license claimed with your suffolk.edu email

**Next week — Session 3:** the model told you *what* to do. Sensitivity analysis tells you *what it's worth*: shadow prices, reduced costs, and the one-more-hour question from Part 1. 🚀